# Training and effective dimension: fully tensorized biased vs. unbiased models

This notebook reproduces the paper's central full-vs-cutoff comparison (Fig. 4) for the fully-tensorized model of this folder, in which both the map `U` (an MPO+TTN isometric mapping of inputs into the correlation space) and `V^T` (a tensor train over the parameters theta) are represented as tensor networks. It loads the pickled results produced by `train_and_FIM_biased.py` and `train_and_FIM_UNbiased.py` — pairs of a "full" model (flat correlation spectrum, high effective dimension) and a "cutoff" model (spectrum decayed past index `cutoff`, lower effective dimension), each trained by MSE minimization — once for a biased data-generating function (exactly reproduced by the cutoff model) and once for an independently-drawn, unbiased one. It then plots MSE training curves for the full and cutoff models in each setting, and finally Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min against the effective-dimension gap d_eff^full - d_eff^cut, illustrating the sign flip between the biased case (positive: low effective dimension trains better) and the unbiased case (negative: high effective dimension trains better).

Imports and path setup for loading custom plotting/analysis utilities.

In [ ]:
# Importing necessary packages
import sys
import os
import copy
import re
import pickle

import pennylane.numpy as np



import matplotlib
import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

Experiment specs (bond dimension, correlation-space dimension, cutoff index, decay exponent, number of features/parameters, local parameter-basis dimension, etc.) matching the training scripts, used to build the filenames of the pickled results to load.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ------------------------------------ Basic model specs ----------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

# Folder in which to load data from
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/train_and_FIM_fullyTN_Dloc3/'

### 'biased_data_gen': the data generating funct. is fully encompassed by both full and cutoff models
### 'UNbiased_data_gen': the data generating funct. is random, not encompassed by full and cutoff models
### 'SLIGHTbiased_data_gen': the data generating funct. is only approximately encompassed by full and cutoff models

name_data_gen_fullbiased = 'biased_data_gen'
name_data_gen_unbiased = 'UNbiased_data_gen'

# Learning rate
learning_rate = 0.02
learning_rate_name = '0p02'

# Batch size for training
batch_size = 12
batch_size_name = str(batch_size)

# Bond dimension
bond_dim = 30
name_bond_dim = str(bond_dim)

### Dimension of correlation space to which input functions are mapped
dim_corr_space = 30
name_corr_space = str(dim_corr_space)

### Cutoff index for correlations among Fourier components
#cutoff = 8; to_add = '_3'
#cutoff = 5; to_add = '_2'
cutoff = 2; to_add = ''
cutoff_name = str(cutoff)

### Decay exponent for cutoff model: 
### S_cut[cutoff+i-1] = np.exp(- decay_exp * i) for i in [1, dim_basis_inputs-cutoff]
#decay_exp = 0.5
#decay_exp_name = '0p5'
#decay_exp = 0.75
#decay_exp_name = '0p75'
decay_exp = 1.0
decay_exp_name = '1p0'

### No. of features (dimension of input vectors)
no_of_features = 4
name_no_features = str(no_of_features)

### Maximal Fourier frequency
max_frequency = 3
name_max_freq = str(max_frequency)

### No. of parameters
no_params = 24
name_no_params = str(no_params)

### No. basis states per parameter
dim_basis_single_param = 3  ### (1, cos(th), sin(th))
name_dim_basis_param = str(dim_basis_single_param)

Load and aggregate the pickled results produced by `train_and_FIM_biased.py` (`biased_data_gen`) into `dict_fullbiased_all`, and by `train_and_FIM_UNbiased.py` (`UNbiased_data_gen`) into `dict_unbiased_all`, keyed by model draw — including MSE/validation-MSE training curves, normalized FIM spectra, and effective dimensions for both the full and cutoff models.

In [ ]:
name_end = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
            '_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_DimCorrSpace' + name_corr_space + 
            '_' + name_data_gen_fullbiased + '_batch' + batch_size_name + '_lr' + learning_rate_name + 
            '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + '_modeldraw')
filename0 = 'dict_results' + name_end
listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(filename0))]
dict_fullbiased_all = dict()
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+).pkl'), filename)
    nm_name = res[0][0]
    nm = int(nm_name)
    dict_d = dict()
    dict_d['delta_minMSE_train'] = []
    dict_d['delta_minMSE_val'] = []
    dict_d['MSE_full'] = []
    dict_d['MSEval_full'] = []
    dict_d['MSE_cut'] = []
    dict_d['MSEval_cut'] = []
    dict_d['mean_normFIM_spectra_full'] = []
    dict_d['std_normFIM_spectra_full'] = []
    dict_d['min_normFIM_spectra_full'] = []
    dict_d['max_normFIM_spectra_full'] = []
    dict_d['norm_eff_dim_full'] = []
    dict_d['mean_normFIM_spectra_cut'] = []
    dict_d['std_normFIM_spectra_cut'] = []
    dict_d['min_normFIM_spectra_cut'] = []
    dict_d['max_normFIM_spectra_cut'] = []
    dict_d['norm_eff_dim_cut'] = []
    dict_fullbiased_all[nm] = dict_d
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+).pkl'), filename)
    nm_name = res[0][0]
    nm = int(nm_name)
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_model = pickle.load(f)
    dict_d = dict_fullbiased_all[nm]
    dict_d['delta_minMSE_train'].append(dict_model['delta_minMSE_full_m_cut'])
    dict_d['delta_minMSE_val'].append(dict_model['delta_minMSEval_full_m_cut'])
    dict_d['MSE_full'].append(dict_model['MSE_full'])
    dict_d['MSEval_full'].append(dict_model['MSEval_full'])
    dict_d['MSE_cut'].append(dict_model['MSE_cut'])
    dict_d['MSEval_cut'].append(dict_model['MSEval_cut'])
    dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
    dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
    dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
    dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
    dict_d['norm_eff_dim_full'].append(np.squeeze(np.asarray(dict_model['norm_eff_dim_full'])))
    dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
    dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
    dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
    dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
    dict_d['norm_eff_dim_cut'].append(np.squeeze(np.asarray(dict_model['norm_eff_dim_cut'])))
    dict_fullbiased_all[nm] = dict_d


name_end = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + 
            '_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_DimCorrSpace' + name_corr_space + 
            '_' + name_data_gen_unbiased + '_batch' + batch_size_name + '_lr' + learning_rate_name + 
            '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + '_modeldraw')
filename0 = 'dict_results' + name_end
listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(filename0))]
dict_unbiased_all = dict()
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+).pkl'), filename)
    nm_name = res[0][0]
    nm = int(nm_name)
    dict_d = dict()
    dict_d['delta_minMSE_train'] = []
    dict_d['delta_minMSE_val'] = []
    dict_d['MSE_full'] = []
    dict_d['MSEval_full'] = []
    dict_d['MSE_cut'] = []
    dict_d['MSEval_cut'] = []
    dict_d['mean_normFIM_spectra_full'] = []
    dict_d['std_normFIM_spectra_full'] = []
    dict_d['min_normFIM_spectra_full'] = []
    dict_d['max_normFIM_spectra_full'] = []
    dict_d['norm_eff_dim_full'] = []
    dict_d['mean_normFIM_spectra_cut'] = []
    dict_d['std_normFIM_spectra_cut'] = []
    dict_d['min_normFIM_spectra_cut'] = []
    dict_d['max_normFIM_spectra_cut'] = []
    dict_d['norm_eff_dim_cut'] = []
    dict_unbiased_all[nm] = dict_d
for filename in listallfiles:
    res = re.findall((filename0 + '(\S+).pkl'), filename)
    nm_name = res[0][0]
    nm = int(nm_name)
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_model = pickle.load(f)
    dict_d = dict_unbiased_all[nm]
    dict_d['delta_minMSE_train'].append(dict_model['delta_minMSE_full_m_cut'])
    dict_d['delta_minMSE_val'].append(dict_model['delta_minMSEval_full_m_cut'])
    dict_d['MSE_full'].append(dict_model['MSE_full'])
    dict_d['MSEval_full'].append(dict_model['MSEval_full'])
    dict_d['MSE_cut'].append(dict_model['MSE_cut'])
    dict_d['MSEval_cut'].append(dict_model['MSEval_cut'])
    dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
    dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
    dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
    dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
    dict_d['norm_eff_dim_full'].append(np.squeeze(np.asarray(dict_model['norm_eff_dim_full'])))
    dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
    dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
    dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
    dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
    dict_d['norm_eff_dim_cut'].append(np.squeeze(np.asarray(dict_model['norm_eff_dim_cut'])))
    dict_unbiased_all[nm] = dict_d

MSE training curves (each individual training run shown separately, for one model draw) for the biased full vs. cutoff model, labeled with each model's effective dimension.

In [ ]:
n_draw = 2
dict_data = dict_fullbiased_all
to_plot = 'MSE'
#to_plot = 'MSEval'

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_full']))
yall_f = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_full']))

ned_c = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_cut']))
yall_c = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_cut']))

x_f = np.arange(1, yall_f.shape[1] + 1)
ax.plot(x_f, yall_f[0, :], 'o-', color='tab:blue', markersize=5, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
for i in range(1, yall_f.shape[0]):
    ax.plot(x_f, yall_f[i, :], 'o-', color='tab:blue', markersize=5, linewidth=3)
#ax.fill_between(x_f, min_y_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, yall_c.shape[1] + 1)
ax.plot(x_c, yall_c[0, :], 'o-', color='tab:orange', markersize=5, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
for i in range(1, yall_c.shape[0]):
    ax.plot(x_c, yall_c[i, :], 'o-', color='tab:orange', markersize=5, linewidth=3)
#ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

MSE training curves (each individual training run shown separately, for one model draw) for the unbiased full vs. cutoff model, labeled with each model's effective dimension.

In [ ]:
n_draw = 2
dict_data = dict_unbiased_all
to_plot = 'MSE'
#to_plot = 'MSEval'

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_full']))
yall_f = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_full']))

ned_c = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_cut']))
yall_c = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_cut']))

x_f = np.arange(1, yall_f.shape[1] + 1)
ax.plot(x_f, yall_f[0, :], 'o-', color='tab:blue', markersize=5, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
for i in range(1, yall_f.shape[0]):
    ax.plot(x_f, yall_f[i, :], 'o-', color='tab:blue', markersize=5, linewidth=3)
#ax.fill_between(x_f, min_y_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, yall_c.shape[1] + 1)
ax.plot(x_c, yall_c[0, :], 'o-', color='tab:orange', markersize=5, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
for i in range(1, yall_c.shape[0]):
    ax.plot(x_c, yall_c[i, :], 'o-', color='tab:orange', markersize=5, linewidth=3)
#ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

Mean MSE training curve (shaded band spanning the min/max across training runs) for the biased full vs. cutoff model.

In [ ]:
n_draw = 2
dict_data = dict_fullbiased_all
to_plot = 'MSE'
#to_plot = 'MSEval'

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_full']))
yall_f = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_full']))

ned_c = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_cut']))
yall_c = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_cut']))

y_f = np.mean(yall_f, axis=0)
min_ymin_f = np.min(yall_f, axis=0)
max_y_f = np.max(yall_f, axis=0)
y_c = np.mean(yall_c, axis=0)
min_y_c = np.min(yall_c, axis=0)
max_y_c = np.max(yall_c, axis=0)

x_f = np.arange(1, yall_f.shape[1] + 1)
ax.plot(x_f, y_f, 'o-', color='tab:blue', markersize=7, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
ax.fill_between(x_f, min_ymin_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, len(y_c) + 1)
ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

Save the biased full-vs-cutoff mean MSE training plot to disk (PNG and PDF).

In [ ]:
fig.savefig('MSE_training_fullbiased_full_and_cut' + to_add + '.png', bbox_inches='tight')
fig.savefig('MSE_training_fullbiased_full_and_cut' + to_add + '.pdf', bbox_inches='tight')

Mean MSE training curve (shaded band spanning the min/max across training runs) for the unbiased full vs. cutoff model.

In [ ]:
n_draw = 2
dict_data = dict_unbiased_all
to_plot = 'MSE'
#to_plot = 'MSEval'

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_full']))
yall_f = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_full']))

ned_c = np.squeeze(np.asarray(dict_data[n_draw]['norm_eff_dim_cut']))
yall_c = np.squeeze(np.asarray(dict_data[n_draw][to_plot + '_cut']))

y_f = np.mean(yall_f, axis=0)
min_ymin_f = np.min(yall_f, axis=0)
max_y_f = np.max(yall_f, axis=0)
y_c = np.mean(yall_c, axis=0)
min_y_c = np.min(yall_c, axis=0)
max_y_c = np.max(yall_c, axis=0)

x_f = np.arange(1, yall_f.shape[1] + 1)
ax.plot(x_f, y_f, 'o-', color='tab:blue', markersize=7, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
ax.fill_between(x_f, min_ymin_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, len(y_c) + 1)
ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='upper right', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

Save the unbiased full-vs-cutoff mean MSE training plot to disk (PNG and PDF).

In [ ]:
fig.savefig('MSE_training_unbiased_full_and_cut' + to_add + '.png', bbox_inches='tight')
fig.savefig('MSE_training_unbiased_full_and_cut' + to_add + '.pdf', bbox_inches='tight')

Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min vs. the effective-dimension gap d_eff^full - d_eff^cut, with biased (dark) and unbiased (light) draws overlaid, showing the sign flip between the two regimes.

In [ ]:
fs = 28
figsize = (9,6)

cmap = matplotlib.colormaps['cividis']

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

deltaMSE_all_fullbiased = []
deltaMSEval_all_fullbiased = []
nED_full_all_fullbiased = []
nED_cut_all_fullbiased = []
for nm in dict_fullbiased_all:
    dict_dec = dict_fullbiased_all[nm]
    nED_full = np.squeeze(np.asarray(dict_dec['norm_eff_dim_full'][0]))
    nED_cut = np.squeeze(np.asarray(dict_dec['norm_eff_dim_cut'][0]))
    deltaMSE = np.squeeze(np.asarray(dict_dec['delta_minMSE_train'][0]))
    deltaMSEval = np.squeeze(np.asarray(dict_dec['delta_minMSE_val'][0]))
    deltaMSE_all_fullbiased.append(deltaMSE)
    deltaMSEval_all_fullbiased.append(deltaMSEval)
    print(deltaMSE.shape)
    no_tries = len(deltaMSE)
    nED_full_vec = np.asarray([nED_full for _ in range(no_tries)])
    nED_cut_vec = np.asarray([nED_cut for _ in range(no_tries)])
    nED_full_all_fullbiased.append(nED_full_vec)
    nED_cut_all_fullbiased.append(nED_cut_vec)
nED_full_fullbiased = np.concatenate(nED_full_all_fullbiased)
nED_cut_fullbiased = np.concatenate(nED_cut_all_fullbiased)
deltaMSE_all_fullbiased = np.concatenate(deltaMSE_all_fullbiased)
deltaMSEval_all_fullbiased = np.concatenate(deltaMSEval_all_fullbiased)
y = deltaMSE_all_fullbiased
#y = deltaMSEval_all_fullbiased
x = nED_full_fullbiased - nED_cut_fullbiased
ax.plot(x, y, 'D', color=cmap(0.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy1 = copy.deepcopy(y)

deltaMSE_all_unbiased = []
deltaMSEval_all_unbiased = []
nED_full_all_unbiased = []
nED_cut_all_unbiased = []
for nm in dict_unbiased_all:
    dict_dec = dict_unbiased_all[nm]
    nED_full = np.squeeze(np.asarray(dict_dec['norm_eff_dim_full'][0]))
    nED_cut = np.squeeze(np.asarray(dict_dec['norm_eff_dim_cut'][0]))
    deltaMSE = np.squeeze(np.asarray(dict_dec['delta_minMSE_train'][0]))
    deltaMSEval = np.squeeze(np.asarray(dict_dec['delta_minMSE_val'][0]))
    deltaMSE_all_unbiased.append(deltaMSE)
    deltaMSEval_all_unbiased.append(deltaMSEval)
    no_tries = len(deltaMSE)
    nED_full_vec = np.asarray([nED_full for _ in range(no_tries)])
    nED_cut_vec = np.asarray([nED_cut for _ in range(no_tries)])
    nED_full_all_unbiased.append(nED_full_vec)
    nED_cut_all_unbiased.append(nED_cut_vec)
nED_full_unbiased = np.concatenate(nED_full_all_unbiased)
nED_cut_unbiased = np.concatenate(nED_cut_all_unbiased)
deltaMSE_all_unbiased = np.concatenate(deltaMSE_all_unbiased)
deltaMSEval_all_unbiased = np.concatenate(deltaMSEval_all_unbiased)
y = deltaMSE_all_unbiased
#y = deltaMSEval_all_unbiased
x = nED_full_unbiased - nED_cut_unbiased
ax.plot(x, y, 'D', color=cmap(1.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy2 = copy.deepcopy(y)

xx = np.arange(0.286, 0.32, 0.001)
ax.plot(xx, np.zeros(len(xx)), '--', color='red', linewidth=3)

#ax.legend(fontsize=22)
ax.set_xlabel(r'$\hat{d}^{\mathrm{full}}_{\mathrm{eff}}-\hat{d}^{\mathrm{cut}}_{\mathrm{eff}}$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_yscale('symlog', linthresh=0.01)
#ax.set_ylim([1.0e-12, 1.0e03])

Save the Delta_{f-c}MSE_min vs. effective-dimension-gap plot to disk (PNG and PDF).

In [ ]:
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_biased_and_unbiased' + to_add + '.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_biased_and_unbiased' + to_add + '.pdf', bbox_inches='tight')